In [4]:
pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install trl  sentencepiece peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 8.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]1/2 [trl]
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # only one GPU

In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

False
0


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
import torch
from scipy.special import softmax
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,BatchNormalization, Activation
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

In [2]:
data = pd.read_csv("/kaggle/input/datasets/rizwanaliza/alzheimers-text-data/alzhermer_text_data.csv")

data.head(5)

,text,Alzheimer’s Diagnosis
0,The patient's age is 90.The patient's Employme...,No
1,The patient's age is 72.The patient's Employme...,No
2,The patient's age is 86.The patient's Employme...,No
3,The patient's age is 53.The patient's Employme...,No
4,The patient's age is 58.The patient's Employme...,No


In [3]:
data.isnull().sum()

text                     0
Alzheimer’s Diagnosis    0
dtype: int64

In [ ]:
import transformers
from transformers import pipeline,set_seed
from transformers import DataCollatorWithPadding
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import re
from tqdm import tqdm
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "prithivMLmods/Llama-Chat-Summary-3.2-3B"
summary_tokenizer = AutoTokenizer.from_pretrained(model_name)
summary_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
    device_map = "auto"
)


In [ ]:
print(data["text"].dtype)
print(data["text"].head())
print(data["text"].isna().sum())
print(data["text"].apply(type).value_counts())

In [ ]:
def summarize(patient_text):

    prompt = f"""Summarize the following patient record in concise natural language.

Rules:
- Preserve every patient feature and its value.
- Do not add facts that are not provided.
- Do not predict or mention Alzheimer's diagnosis.
- Do not change or infer medical conditions.

Patient record:
{patient_text}

Summary:
"""

    inputs = summary_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    device = next(summary_model.parameters()).device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    pad_token_id = (
    summary_tokenizer.pad_token_id
    if summary_tokenizer.pad_token_id is not None
    else summary_tokenizer.eos_token_id
    )

    eos_token_id = summary_tokenizer.eos_token_id
 

    with torch.inference_mode():
        outputs = summary_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            num_beams=1,
            use_cache = True,
            pad_token_id = pad_token_id,
            eos_token_id = eos_token_id
            
    )  
    newtokens = outputs[0, inputs["input_ids"].shape[1]:]
    summary = summary_tokenizer.decode(
        newtokens,
        skip_special_tokens=True
    ).strip()

    return summary

In [ ]:
data['summarize'] = df['text'].map(summarize)